In [49]:
import json
from collections import Counter
from pathlib import Path
import pandas as pd

In [44]:
# EDIT THESE TWO PATHS
SRC_ROOT = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/raw_labels")  # the folder containing train/ val/ test/
OUT_DIR = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val")

In [14]:
#Folder check 
for s in ["train", "val", "test"]:
    n = len(list((SRC_ROOT / s).glob("*.json"))) #glob finds all path names matching specific pattern and returns Generator object 
    print(s, n)

train 70000
val 10000
test 20000


In [16]:
#Peeking at one file

file = sorted((SRC_ROOT / "train").glob("*.json"))[0]
file_content = json.load(open(file))
print("File name :", file.name)
print("Json Keys :", list(file_content.keys()))
print("Json attributes :", file_content.get("attributes"))

File name : 0000f77c-6257be58.json
Json Keys : ['name', 'frames', 'attributes']
Json attributes : {'weather': 'clear', 'scene': 'city street', 'timeofday': 'daytime'}


In [37]:
path = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/raw_labels/train/0a0a0b1a-7c39d841.json")
with open(path) as fh:
    d = json.load(fh)
    items, weather, n_boxes, n_no_attr = [], Counter(), 0, 0

    name = d.get("name") or f.stem
    if not name.endswith(".jpg"):
        name += ".jpg"

        # per-image files store objects under frames[0]["objects"] (or "labels")
    objs = d["frames"][0].get("objects", []) 

    labels = [
        {"category": o["category"], "box2d": o["box2d"], "attributes": o.get("attributes", {})}
        for o in objs
        if o.get("box2d") is not None  # skip lanes / drivable area
        ]
    attrs = d.get("attributes") or {}
    n_no_attr += not attrs
    weather[attrs.get("weather", "missing")] += 1
    print(weather)
    n_boxes += len(labels)
    items.append({"name": name, "attributes": attrs, "labels": labels})
    print(items)


Counter({'clear': 1})
[{'name': '0a0a0b1a-7c39d841.jpg', 'attributes': {'weather': 'clear', 'scene': 'highway', 'timeofday': 'daytime'}, 'labels': [{'category': 'car', 'box2d': {'x1': 555.647397, 'y1': 304.228432, 'x2': 574.015906, 'y2': 316.474104}, 'attributes': {'occluded': True, 'truncated': False, 'trafficLightColor': 'none'}}, {'category': 'car', 'box2d': {'x1': 554.116689, 'y1': 318.004813, 'x2': 567.89307, 'y2': 328.719775}, 'attributes': {'occluded': True, 'truncated': False, 'trafficLightColor': 'none'}}, {'category': 'car', 'box2d': {'x1': 523.502507, 'y1': 311.881977, 'x2': 544.932435, 'y2': 327.189068}, 'attributes': {'occluded': True, 'truncated': False, 'trafficLightColor': 'none'}}, {'category': 'car', 'box2d': {'x1': 492.888326, 'y1': 311.881977, 'x2': 526.563926, 'y2': 337.904031}, 'attributes': {'occluded': True, 'truncated': False, 'trafficLightColor': 'none'}}, {'category': 'truck', 'box2d': {'x1': 393.392235, 'y1': 302.697723, 'x2': 453.089891, 'y2': 347.088286}, 

In [46]:
# To merge all jsons into one json

def convert_split(src_dir: Path, out_path: Path):
    files = sorted(src_dir.glob("*.json"))
    assert files, f"No .json files found in {src_dir}"

    items, weather, n_boxes, n_no_attr = [], Counter(), 0, 0
    for f in files:
        with open(f) as fh:
            d = json.load(fh)

        name = d.get("name") or f.stem
        if not name.endswith(".jpg"):
            name += ".jpg"

        # per-image files store objects under frames[0]["objects"] (or "labels")
        objs = d["frames"][0].get("objects", []) if d.get("frames") else d.get("labels", [])

        labels = [
            {"category": o["category"], "box2d": o["box2d"], "attributes": o.get("attributes", {})}
            for o in objs
            if o.get("box2d") is not None  # skip lanes / drivable area
        ]
        attrs = d.get("attributes") or {}
        n_no_attr += not attrs
        weather[attrs.get("weather", "missing")] += 1
        n_boxes += len(labels)
        items.append({"name": name, "attributes": attrs, "labels": labels})

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as fh:
        json.dump(items, fh)

    print(f"{src_dir.name}: {len(items)} images, {n_boxes} boxes -> {out_path}")
    print(f"  weather values: {dict(weather)}")
    if n_no_attr:
        print(f"  WARNING: {n_no_attr} files had no image-level attributes")


for split in SPLITS:
    convert_split(SRC_ROOT / split, OUT_DIR / f"bdd100k_labels_images_{split}.json")

FileExistsError: [Errno 17] File exists: '/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val'

In [45]:
path = Path("/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/raw_labels/val")
convert_split(path,OUT_DIR)
SPLITS = ["val"]

val: 10000 images, 185578 boxes -> /Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val
  weather values: {'overcast': 1239, 'clear': 5346, 'undefined': 1157, 'rainy': 738, 'snowy': 769, 'partly cloudy': 738, 'foggy': 13}


In [80]:
def summarize_conditions(train: list[dict], val: list[dict]) -> pd.DataFrame:
    """Count images per weather / scene / time of day for each split."""
    rows = []
    for split, items in [("train", train), ("val", val)]:
        print("split",split)
        print("items",items)
        n = len(items) # no of images
        print('n:', n)
        for key in ["weather", "scene", "timeofday"]:
            counts = Counter(x["attributes"].get(key, "missing") for x in items)
            for value, c in counts.most_common():
                print(value , " : " ,c)
                rows.append(
                    {"split": split, "attribute": key, "value": value,
                     "images": c, "pct": round(100 * c / n, 1)}
                )
    return pd.DataFrame(rows)

In [81]:
val_path = "/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val1.json"
train_path = "/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/train1.json"
train = json.load(open(train_path))
val = json.load(open(val_path))
summarize_conditions(train, val)

split train
items [{'name': 'b1c66a42-6f7d68ca.jpg', 'attributes': {'weather': 'overcast', 'scene': 'city street', 'timeofday': 'daytime'}, 'labels': [{'category': 'traffic sign', 'box2d': {'x1': 652.575363, 'y1': 303.204232, 'x2': 685.016968, 'y2': 315.681772}, 'attributes': {'occluded': False, 'truncated': False, 'trafficLightColor': 'none'}}]}, {'name': 'b1c81faa-3df17267.jpg', 'attributes': {'weather': 'clear', 'scene': 'highway', 'timeofday': 'night'}, 'labels': [{'category': 'car', 'box2d': {'x1': 819.464053, 'y1': 280.082505, 'x2': 889.23726, 'y2': 312.742305}, 'attributes': {'occluded': False, 'truncated': False, 'trafficLightColor': 'none'}}, {'category': 'car', 'box2d': {'x1': 1027.299139, 'y1': 290.47426, 'x2': 1052.536255, 'y2': 306.804159}, 'attributes': {'occluded': False, 'truncated': False, 'trafficLightColor': 'none'}}]}]
n: 2
overcast  :  1
clear  :  1
city street  :  1
highway  :  1
daytime  :  1
night  :  1
split val
items [{'name': 'b1c66a42-6f7d68ca.jpg', 'attribu

,split,attribute,value,images,pct
0,train,weather,overcast,1,50.0
1,train,weather,clear,1,50.0
2,train,scene,city street,1,50.0
3,train,scene,highway,1,50.0
4,train,timeofday,daytime,1,50.0
5,train,timeofday,night,1,50.0
6,val,weather,overcast,1,50.0
7,val,weather,clear,1,50.0
8,val,scene,city street,1,50.0
9,val,scene,highway,1,50.0


In [74]:
val_path = "/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val1.json"

val = json.load(open(val_path))

print(type(val), len(val))   # <class 'list'> 10000
print(val[0]["attributes"])

rows = []
n = len(val)
print(val)
for key in ["weather", "scene", "timeofday"]:
    counts = Counter(x["attributes"].get(key, "missing") for x in val)
    for value, c in counts.most_common():
        print(value)
        print("c", c)
        rows.append({"attribute": key, "value": value, "images": c, "pct": round(100 * c / n, 1)})

pd.DataFrame(rows)

<class 'list'> 2
{'weather': 'overcast', 'scene': 'city street', 'timeofday': 'daytime'}
[{'name': 'b1c66a42-6f7d68ca.jpg', 'attributes': {'weather': 'overcast', 'scene': 'city street', 'timeofday': 'daytime'}, 'labels': [{'category': 'traffic sign', 'box2d': {'x1': 652.575363, 'y1': 303.204232, 'x2': 685.016968, 'y2': 315.681772}, 'attributes': {'occluded': False, 'truncated': False, 'trafficLightColor': 'none'}}]}, {'name': 'b1c81faa-3df17267.jpg', 'attributes': {'weather': 'clear', 'scene': 'highway', 'timeofday': 'night'}, 'labels': [{'category': 'car', 'box2d': {'x1': 819.464053, 'y1': 280.082505, 'x2': 889.23726, 'y2': 312.742305}, 'attributes': {'occluded': False, 'truncated': False, 'trafficLightColor': 'none'}}, {'category': 'car', 'box2d': {'x1': 1027.299139, 'y1': 290.47426, 'x2': 1052.536255, 'y2': 306.804159}, 'attributes': {'occluded': False, 'truncated': False, 'trafficLightColor': 'none'}}]}]
overcast
c 1
clear
c 1
city street
c 1
highway
c 1
daytime
c 1
night
c 1


,attribute,value,images,pct
0,weather,overcast,1,50.0
1,weather,clear,1,50.0
2,scene,city street,1,50.0
3,scene,highway,1,50.0
4,timeofday,daytime,1,50.0
5,timeofday,night,1,50.0


PosixPath('/Users/shravanthij/Documents/ML academy/driving-perception-validation/data/01_raw/bdd100k/labels/val1')